In [ ]:
#code for renaming the image name
#kvasir-seg and sessile dataset have same image names but the polyps are different so i appended 1,2,3..... in the suffix of every image and mask to merge both the dataset images and masks.
import os
import shutil

# Input directories
image_dir = r"C:\Medical_image_analysis\colon_cancer\kvasir-sessile\sessile-main-Kvasir-SEG\images"
mask_dir = r"C:\Medical_image_analysis\colon_cancer\kvasir-sessile\sessile-main-Kvasir-SEG\masks"

# Output directories
output_dir_images = r"C:\Medical_image_analysis\kvasir_seg_sessile\sessile\images"
output_dir_masks = r"C:\Medical_image_analysis\kvasir_seg_sessile\sessile\masks"

# Create output dirs if not exist
os.makedirs(output_dir_images, exist_ok=True)
os.makedirs(output_dir_masks, exist_ok=True)

# List image files
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))])

# Append counter and save
for idx, file in enumerate(image_files, 1):
    name, ext = os.path.splitext(file)
    new_name = f"{name}_{idx}{ext}"

    # Copy image
    shutil.copy(
        os.path.join(image_dir, file),
        os.path.join(output_dir_images, new_name)
    )

    # Copy corresponding mask (assumes same name and extension)
    mask_path = os.path.join(mask_dir, file)
    if os.path.exists(mask_path):
        shutil.copy(
            mask_path,
            os.path.join(output_dir_masks, new_name)
        )
    else:
        print(f"Mask not found for: {file}")

print("Renaming and copying done!")

Renaming and copying done!


In [ ]:
# code to convert the masks into the yolo format

import os
import cv2

# Set up paths
image_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\images"
mask_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\masks"
label_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\labels"

# Create the labels folder if it doesn't exist
os.makedirs(label_dir, exist_ok=True)

# Get all image files
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Loop through all images
for file in image_files:
    img_path = os.path.join(image_dir, file)
    mask_path = os.path.join(mask_dir, file)
    label_path = os.path.join(label_dir, os.path.splitext(file)[0] + ".txt")

    # Load image and mask
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    height, width = image.shape[:2]

    # Threshold the mask in case it's not binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find all external contours (each polyp)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    with open(label_path, 'w') as label_file:
        for cnt in contours:
            x, y, w_box, h_box = cv2.boundingRect(cnt)

            # Optional: ignore small bounding boxes (noise)
            if w_box < 5 or h_box < 5:
                continue

            # Convert to YOLO format (normalized)
            x_center = (x + w_box / 2) / width
            y_center = (y + h_box / 2) / height
            norm_w = w_box / width
            norm_h = h_box / height

            # Class ID is 0 (for polyp)
            yolo_line = f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
            label_file.write(yolo_line + "\n")


In [9]:
#code to split the data in train(75%), test(15%), val(15%)

import os
import shutil
import random

# Input directories
image_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\images"
label_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\labels"

# Output directories
base_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile"
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(base_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(base_dir, 'labels', split), exist_ok=True)

# Get image files (assumes .jpg)
image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
image_files.sort()

# Shuffle and split
random.seed(42)
random.shuffle(image_files)

total = len(image_files)
train_end = int(0.7 * total)
val_end = int(0.85 * total)

train_files = image_files[:train_end]
val_files = image_files[train_end:val_end]
test_files = image_files[val_end:]

# Function to move files
def move_files(files, split):
    for file in files:
        name = os.path.splitext(file)[0]
        img_src = os.path.join(image_dir, f"{name}.jpg")
        lbl_src = os.path.join(label_dir, f"{name}.txt")

        img_dst = os.path.join(base_dir, 'images', split, f"{name}.jpg")
        lbl_dst = os.path.join(base_dir, 'labels', split, f"{name}.txt")

        # Copy image and label
        if os.path.exists(img_src) and os.path.exists(lbl_src):
            shutil.copy(img_src, img_dst)
            shutil.copy(lbl_src, lbl_dst)

# Move files
move_files(train_files, 'train')
move_files(val_files, 'val')
move_files(test_files, 'test')

print("✅ Dataset split completed:")
print(f"Train: {len(train_files)} images")
print(f"Val:   {len(val_files)} images")
print(f"Test:  {len(test_files)} images")


✅ Dataset split completed:
Train: 837 images
Val:   179 images
Test:  180 images


In [1]:
from ultralytics import YOLO
import time

# Start timing
start = time.time()

# Load your YOLO model (make sure 'yolo11l.pt' exists in working directory)
model = YOLO("yolo12m.pt")

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml",  # path to your data.yaml
    epochs=500,
    amp=False,
    patience=150,
    imgsz=640,
    device=0,
    batch=8,
    name="kvasir_CIoU_yolo12m",                # folder name inside runs/detect/
    project=r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect"  # where results will be saved
)

# End timing
end = time.time()
print(f"✅ Training completed in {(end - start)/60:.2f} minutes.")


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Admin\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [6]:
from ultralytics import YOLO
import time

# Start timing
start = time.time()

# Load your YOLO model (make sure 'yolo11l.pt' exists in working directory)
model = YOLO("yolo12s.pt")

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml",  # path to your data.yaml
    epochs=500,
    amp=False,
    patience=150,
    imgsz=640,
    device=0,
    batch=8,
    name="kvasir_CIoU_yolo12s",                # folder name inside runs/detect/
    project=r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect"  # where results will be saved
)

# End timing
end = time.time()
print(f"✅ Training completed in {(end - start)/60:.2f} minutes.")

100%|██████████| 18.1M/18.1M [00:00<00:00, 44.5MB/s]

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=kvasir_CIoU_yolo12s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=150, persp

  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  2    689408  ultralytics.nn.modules.block.A2C2f           [256, 256, 2, True, 4]        
  7                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  8                  -1  2   2689536  ultralytics.nn.modules.block.A2C2f           [512, 512, 2, True, 1]        
  9                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None

train: Scanning C:\Medical_image_analysis\kvasir_seg_sessile\labels\train.cache... 837 images, 0 backgrounds, 0 corrupt: 100%|██████████| 837/837 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.20.0 ms, read: 237.268.7 MB/s, size: 42.5 KB)


...

    500/500      7.51G     0.2614     0.1702     0.8467          5        640: 100%|██████████| 105/105 [00:16<00:00,  6.19it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:01<00:00,  6.91it/s]

                   all        179        189      0.925      0.825      0.911      0.778



500 epochs completed in 2.746 hours.
Optimizer stripped from C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_CIoU_yolo12s\weights\last.pt, 19.0MB
Optimizer stripped from C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_CIoU_yolo12s\weights\best.pt, 19.0MB

Validating C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_CIoU_yolo12s\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:01<00:00,  6.11it/s]


                   all        179        189       0.95      0.841      0.929      0.793
Speed: 0.3ms preprocess, 4.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_CIoU_yolo12s
✅ Training completed in 166.15 minutes.


In [7]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_CIoU_yolo12s\weights\best.pt")

# Validate using the test split defined in the YAML
metrics = model.val(data=r"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml", split='test')

# Extract class-wise metrics (assuming only one class)
precision = metrics.box.p[0]
recall = metrics.box.r[0]
f1_score = metrics.box.f1[0]
map_50 = metrics.box.map50
map_50_95 = metrics.box.map

# Print results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print(f"mAP@0.5: {map_50:.4f}")
print(f"mAP@0.5:0.95: {map_50_95:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 120.130.4 MB/s, size: 29.3 KB)


val: Scanning C:\Medical_image_analysis\kvasir_seg_sessile\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100%|██████████| 180/180 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:02<00:00,  5.07it/s]


                   all        180        187      0.949      0.904      0.949      0.819
Speed: 0.6ms preprocess, 6.1ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs\detect\val3
Precision: 0.9488
Recall: 0.9037
F1-score: 0.9257
mAP@0.5: 0.9495
mAP@0.5:0.95: 0.8191


In [1]:
from ultralytics import YOLO
import time

# Start timing
start = time.time()

# Load your YOLO model (make sure 'yolo11l.pt' exists in working directory)
model = YOLO("yolo12m.pt")

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml",  # path to your data.yaml
    epochs=500,
    amp=False,
    patience=150,
    imgsz=640,
    device=0,
    batch=8,
    name="kvasir_CIoU_yolo12m",                # folder name inside runs/detect/
    project=r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect"  # where results will be saved
)

# End timing
end = time.time()
print(f"✅ Training completed in {(end - start)/60:.2f} minutes.")

RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [5]:
from ultralytics import YOLO

model = YOLO(r"C:\Medical_image_analysis\colon_cancer\yolo12m.pt")


In [6]:
model.train(
    data=r"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    save=True,
    project=r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect"
)


New https://pypi.org/project/ultralytics/8.3.154 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Medical_image_analysis\colon_cancer\yolo12m.pt, momentum=0.937, mosaic=1.0, mul

100%|██████████| 5.35M/5.35M [00:00<00:00, 67.4MB/s]


AMP: checks passed 
train: Fast image access  (ping: 0.40.1 ms, read: 59.58.7 MB/s, size: 36.0 KB)


train: Scanning C:\Medical_image_analysis\kvasir_seg_sessile\labels\train.cache... 837 images, 0 backgrounds, 0 corrupt: 100%|██████████| 837/837 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.30.1 ms, read: 75.818.6 MB/s, size: 42.5 KB)


...

                   all        179        189      0.916      0.884      0.914      0.782



500 epochs completed in 2.905 hours.
Optimizer stripped from C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\train\weights\last.pt, 40.8MB
Optimizer stripped from C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\train\weights\best.pt, 40.8MB

Validating C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\train\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:02<00:00,  2.78it/s]


                   all        179        189      0.938      0.885      0.939      0.808
Speed: 0.1ms preprocess, 5.7ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000023190C758A0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [2]:
import os

print(os.path.exists("yolo12m.pt"))  # Should return True


True


In [3]:
import os

file_path = "yolo12m.pt"
if os.path.exists(file_path):
    print(f"{file_path} exists, size: {os.path.getsize(file_path)} bytes")
else:
    print(f"{file_path} does not exist!")


yolo12m.pt exists, size: 10485760 bytes


In [2]:
from ultralytics import YOLO
import os

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_yolo12m_15_064\weights\best.pt")

# Path to your CVC-ClinicDB images
source_folder = r"C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original"

# Run prediction
model.predict(
    source=source_folder,        # Path to folder with images
    save=True,                   # Save predictions
    save_txt=True,               # Save labels in YOLO format
    conf=0.25,                   # Confidence threshold
    imgsz=640,                   # Resize image to 640x640 before prediction
    device='cuda'                # Use GPU (if available), or use 'cpu'
)



image 1/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\1.png: 480x640 1 polyp, 152.4ms
image 2/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\10.png: 480x640 1 polyp, 53.4ms
image 3/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\100.png: 480x640 1 polyp, 21.6ms
image 4/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\101.png: 480x640 1 polyp, 21.0ms
image 5/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\102.png: 480x640 1 polyp, 21.0ms
image 6/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\103.png: 480x640 1 polyp, 20.9ms
image 7/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\104.png: 480x640 1 polyp, 21.9ms
image 8/612 C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original\105.png: 480x640 1 polyp, 21.3ms
image 9/612 C:\Medical_im

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'polyp'}
 obb: None
 orig_img: array([[[11, 11, 11],
         [11, 11, 11],
         [11, 11, 11],
         ...,
         [11, 11, 11],
         [11, 11, 11],
         [11, 11, 11]],
 
        [[11, 11, 11],
         [11, 11, 11],
         [11, 11, 11],
         ...,
         [11, 11, 11],
         [11, 11, 11],
         [11, 11, 11]],
 
        [[11, 11, 11],
         [11, 11, 11],
         [11, 11, 11],
         ...,
         [11, 11, 11],
         [11, 11, 11],
         [11, 11, 11]],
 
        ...,
 
        [[11, 11, 11],
         [11, 11, 11],
         [11, 11, 11],
         ...,
         [11, 11, 11],
         [11, 11, 11],
         [11, 11, 11]],
 
        [[11, 11, 11],
         [11, 11, 11],
         [11, 11, 11],
         ...,
         [11, 11, 11],
         [11, 11, 11],
         [11, 11, 11]],
 
        [[11, 11, 11],
     

In [5]:
from ultralytics import YOLO

model = YOLO(r"C:\Medical_image_analysis\kvasir_seg_sessile\runs\detect\kvasir_yolo12m_15_064\weights\best.pt")

metrics = model.val(data=r"C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\cvc.yaml")

print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 646.8147.5 MB/s, size: 88.1 KB)


val: Scanning C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original.cache... 0 images, 612 backgrounds, 0 corrupt: 100%|██████████| 612/612 [00:00<?, ?it/s]

WARNING Labels are missing or empty in C:\Medical_image_analysis\colon_cancer\CVC-ClinicDB\archive (1)\PNG\Original.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:05<00:00,  7.64it/s]
c:\Users\Admin\anaconda3\envs\yolo_env\lib\site-packages\ultralytics\utils\metrics.py:585: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\Admin\anaconda3\envs\yolo_env\lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Admin\anaconda3\envs\yolo_env\lib\site-packages\ultralytics\utils\metrics.py:630: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
c:\Users\Admin\anaconda3\envs\yolo_env\lib\site-packages\numpy\_core\_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
c:\Users\Admin\anaconda3\envs\yolo_env\lib\site-packages\ultralytics\utils\metrics.py:630: RuntimeWarning: Mean of empty slice

                   all        612          0          0          0          0          0
WARNING no labels found in detect set, can not compute metrics without labels
Speed: 0.2ms preprocess, 6.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs\detect\val10
mAP@0.5: 0.0000
mAP@0.5:0.95: 0.0000


IndexError: index 0 is out of bounds for axis 0 with size 0

In [7]:
# code to convert the masks into the yolo format

import os
import cv2

# Set up paths
image_dir = r"C:\Medical_image_analysis\colon_cancer\kvasir-sessile\sessile-main-Kvasir-SEG\images"
mask_dir = r"C:\Medical_image_analysis\colon_cancer\kvasir-sessile\sessile-main-Kvasir-SEG\masks"
label_dir = r"C:\Medical_image_analysis\colon_cancer\kvasir-sessile\sessile-main-Kvasir-SEG\labels"

# Create the labels folder if it doesn't exist
os.makedirs(label_dir, exist_ok=True)

# Get all image files
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Loop through all images
for file in image_files:
    img_path = os.path.join(image_dir, file)
    mask_path = os.path.join(mask_dir, file)
    label_path = os.path.join(label_dir, os.path.splitext(file)[0] + ".txt")

    # Load image and mask
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    height, width = image.shape[:2]

    # Threshold the mask in case it's not binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find all external contours (each polyp)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    with open(label_path, 'w') as label_file:
        for cnt in contours:
            x, y, w_box, h_box = cv2.boundingRect(cnt)

            # Optional: ignore small bounding boxes (noise)
            if w_box < 5 or h_box < 5:
                continue

            # Convert to YOLO format (normalized)
            x_center = (x + w_box / 2) / width
            y_center = (y + h_box / 2) / height
            norm_w = w_box / width
            norm_h = h_box / height

            # Class ID is 0 (for polyp)
            yolo_line = f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
            label_file.write(yolo_line + "\n")


In [10]:
import cv2
import numpy as np
import os
from glob import glob

input_dir = "C:/Medical_image_analysis/kvasir_seg_sessile/images/train_aug_shrunk"
output_mask_dir = "C:/Medical_image_analysis/lama_masks"
os.makedirs(output_mask_dir, exist_ok=True)

for img_path in glob(os.path.join(input_dir, "*.jpg")):
    img = cv2.imread(img_path)
    mask = np.all(img == 0, axis=2).astype(np.uint8) * 255  # black areas
    filename = os.path.basename(img_path)
    cv2.imwrite(os.path.join(output_mask_dir, filename), mask)


In [ ]:
"C:\Medical_image_analysis\kvasir_seg_sessile\data.yaml"

In [4]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\kvasir-seg_augmentation_20_percent_negative_clahe_yolov8\yolov8_l_20_aug_clahe_7_07\weights\best.pt")

# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\clahe_20_negative_augmentation\data_kvasir.yaml",
    split='val',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 870.5272.9 MB/s, size: 126.6 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\clahe_20_negative_augmentation\labels\val.cache... 241 images, 41 backgrounds, 0 corrupt: 100%|██████████| 241/241 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:04<00:00,  3.73it/s]


                   all        241        214      0.955      0.886      0.959       0.81
Speed: 0.6ms preprocess, 9.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs\detect\val23
mAP@0.5: 0.9591
mAP@0.5:0.95: 0.8096
Precision: 0.9547
Recall: 0.8862
F1-score: 0.9192


In [ ]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\results for kavsir-seg dataset with 10 percent negative samples\yolov8l_kvasir_train_25_06\weights\best.pt")

# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 123.278.8 MB/s, size: 60.6 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\val.cache... 200 images, 20 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:03<00:00,  3.77it/s]


                   all        220        214      0.918      0.893      0.955      0.815
Speed: 0.5ms preprocess, 8.0ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs\detect\val27
mAP@0.5: 0.9553
mAP@0.5:0.95: 0.8147
Precision: 0.9183
Recall: 0.8929
F1-score: 0.9054


In [1]:
#training with dataset bkai with clahe only no augmentation no negative

from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO("yolov8m.pt")  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_m_augmentation_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results",
    workers=4,
    verbose=True,
    patience=50
)

New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_m_augmentation

train: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\train.cache... 7700 images, 700 backgrounds, 0 corrupt: 100%|██████████| 7700/7700 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.30.1 ms, read: 117.622.3 MB/s, size: 37.3 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\val.cache... 200 images, 20 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]


Plotting labels to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        220        214      0.955      0.883      0.947      0.801
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 88, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



138 epochs completed in 4.874 hours.
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\last.pt, 52.0MB
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\best.pt, 52.0MB

Validating C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:03<00:00,  2.28it/s]


                   all        220        214      0.947      0.907      0.965       0.82
Speed: 0.2ms preprocess, 3.8ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D38EF8F250>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [1]:
#training with dataset bkai with clahe only no augmentation no negative

from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO("yolov8s.pt")  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_s_augmentation_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results",
    workers=4,
    verbose=True,
    patience=50
)

New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_s_augmentation

train: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\train.cache... 7700 images, 700 backgrounds, 0 corrupt: 100%|██████████| 7700/7700 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.40.0 ms, read: 52.17.2 MB/s, size: 37.3 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\val.cache... 200 images, 20 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]


Plotting labels to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        220        214      0.941      0.897      0.951      0.791
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 77, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



127 epochs completed in 2.920 hours.
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\last.pt, 22.5MB
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\best.pt, 22.5MB

Validating C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:02<00:00,  2.79it/s]


                   all        220        214      0.966      0.874      0.963      0.809
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EB8893DED0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [24]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_kvasir-seg_dataset_with_augmentations_20_negative\yolov12_s_kvasir_with_augmen_20_negative\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\2_neg\yolo12_kvasir.yaml",
    split='val',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 415.0375.2 MB/s, size: 63.4 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\2_neg\labels\val.cache... 241 images, 41 backgrounds, 0 corrupt: 100%|██████████| 241/241 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:02<00:00,  6.21it/s]


                   all        241        214       0.93      0.902      0.955      0.823
Speed: 0.4ms preprocess, 4.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val62
mAP@0.5: 0.9551
mAP@0.5:0.95: 0.8229
Precision: 0.9299
Recall: 0.9019
F1-score: 0.9157


In [ ]:
from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO('yolov8s.pt')  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_s_kvasir_no_aug_no_negative_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\yolov8_s_kvasir_no_aug_no_negative_CIoU",
    workers=4,
    verbose=True,
    patience=50
)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO('yolov8n.pt')  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_n_kvasir_no_aug_no_negative_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\yolov8_n_kvasir_no_aug_no_negative_CIoU",
    workers=4,
    verbose=True,
    patience=50
)

In [ ]:
#CIoU + No augmentations + No negatives

In [9]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 292.666.4 MB/s, size: 39.5 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:16<00:00,  2.40s/it]


                   all        100        114      0.937      0.807      0.858      0.703
Speed: 0.9ms preprocess, 120.3ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\val71
mAP@0.5: 0.8585
mAP@0.5:0.95: 0.7032
Precision: 0.9371
Recall: 0.8070
F1-score: 0.8672


In [10]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 295.271.6 MB/s, size: 36.6 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:14<00:00,  2.14s/it]


                   all        100        114      0.858      0.795      0.828      0.696
Speed: 0.8ms preprocess, 97.1ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\val72
mAP@0.5: 0.8276
mAP@0.5:0.95: 0.6958
Precision: 0.8580
Recall: 0.7948
F1-score: 0.8252


In [11]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 312.991.7 MB/s, size: 35.8 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.11it/s]


                   all        100        114      0.893      0.833      0.858       0.73
Speed: 0.9ms preprocess, 6.7ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs\detect\val73
mAP@0.5: 0.8581
mAP@0.5:0.95: 0.7298
Precision: 0.8929
Recall: 0.8333
F1-score: 0.8621


In [12]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 298.756.1 MB/s, size: 32.6 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.15it/s]


                   all        100        114      0.905      0.831      0.868       0.72
Speed: 0.8ms preprocess, 4.5ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs\detect\val74
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7197
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


In [7]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 294.876.5 MB/s, size: 36.8 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.10it/s]


                   all        100        114      0.905      0.831      0.868       0.72
Speed: 0.7ms preprocess, 6.8ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs\detect\val97
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7197
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


## YOLOv12 Kvasir-seg (CIoU + No Augmentations + No Negatives) for 100 test images

In [13]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 276.530.5 MB/s, size: 31.0 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:17<00:00,  2.52s/it]


                   all        100        114      0.937      0.807      0.858      0.703
Speed: 0.7ms preprocess, 120.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\val75
mAP@0.5: 0.8585
mAP@0.5:0.95: 0.7032
Precision: 0.9371
Recall: 0.8070
F1-score: 0.8672


In [14]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 317.867.1 MB/s, size: 34.4 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:15<00:00,  2.16s/it]


                   all        100        114      0.858      0.795      0.828      0.696
Speed: 0.7ms preprocess, 98.1ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\val76
mAP@0.5: 0.8276
mAP@0.5:0.95: 0.6958
Precision: 0.8580
Recall: 0.7948
F1-score: 0.8252


In [15]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 279.741.0 MB/s, size: 36.1 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.14it/s]


                   all        100        114      0.893      0.833      0.858       0.73
Speed: 0.7ms preprocess, 5.7ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to runs\detect\val77
mAP@0.5: 0.8581
mAP@0.5:0.95: 0.7298
Precision: 0.8929
Recall: 0.8333
F1-score: 0.8621


In [16]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 251.659.3 MB/s, size: 33.0 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.13it/s]


                   all        100        114      0.905      0.831      0.868       0.72
Speed: 1.0ms preprocess, 4.8ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to runs\detect\val78
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7197
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


In [11]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 274.647.2 MB/s, size: 34.5 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.16it/s]


                   all        100        114      0.942      0.825      0.874       0.73
Speed: 0.8ms preprocess, 3.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs\detect\val101
mAP@0.5: 0.8738
mAP@0.5:0.95: 0.7301
Precision: 0.9418
Recall: 0.8246
F1-score: 0.8793


## YOLOv8 Kvasir-seg (CIoU + No Augmentations + No Negatives) for 100 test images

In [3]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_l_kvasir_no_aug_no_negative_CIoU\yolov8_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 254.646.3 MB/s, size: 28.5 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:39<00:00,  5.63s/it]


                   all        100        114      0.894      0.833      0.845      0.693
Speed: 4.6ms preprocess, 335.0ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\val79
mAP@0.5: 0.8448
mAP@0.5:0.95: 0.6929
Precision: 0.8938
Recall: 0.8333
F1-score: 0.8625


In [4]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_m_kvasir_no_aug_no_negative_CIoU\yolov8_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 449.5331.8 MB/s, size: 57.4 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:23<00:00,  3.30s/it]


                   all        100        114      0.902      0.842       0.86      0.713
Speed: 3.0ms preprocess, 174.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs\detect\val80
mAP@0.5: 0.8601
mAP@0.5:0.95: 0.7133
Precision: 0.9017
Recall: 0.8421
F1-score: 0.8709


In [5]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_s_kvasir_no_aug_no_negative_CIoU\yolov8_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 283.1103.1 MB/s, size: 32.1 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:13<00:00,  1.86s/it]


                   all        100        114      0.905      0.835      0.876      0.723
Speed: 3.0ms preprocess, 75.5ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs\detect\val81
mAP@0.5: 0.8759
mAP@0.5:0.95: 0.7234
Precision: 0.9050
Recall: 0.8353
F1-score: 0.8688


In [6]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 285.947.5 MB/s, size: 34.2 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:08<00:00,  1.16s/it]


                   all        100        114      0.942      0.825      0.874       0.73
Speed: 1.8ms preprocess, 24.9ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\val82
mAP@0.5: 0.8738
mAP@0.5:0.95: 0.7301
Precision: 0.9418
Recall: 0.8246
F1-score: 0.8793


## YOLOv8 (CIoU + No Augmentations + No Negatives) for 100 test images ................(BKAI-IGH NeoPolyp Dataset)...................

In [8]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_l_baseline_model_CIoU\yolov8_l_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,608,150 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 904.6230.3 MB/s, size: 322.8 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:35<00:00,  5.04s/it]


                   all        100        115      0.793      0.805      0.813      0.694
            neoplastic         68         70       0.86        0.9       0.93      0.832
        non-neoplastic         39         45      0.727      0.709      0.696      0.556
Speed: 4.6ms preprocess, 295.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val83
mAP@0.5: 0.8131
mAP@0.5:0.95: 0.6937
Precision: 0.8596
Recall: 0.9000
F1-score: 0.8793


In [9]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_m_baseline_model_CIoU\yolov8_m_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 1229.0241.0 MB/s, size: 265.0 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:19<00:00,  2.74s/it]


                   all        100        115       0.82      0.761      0.811      0.711
            neoplastic         68         70      0.897        0.9      0.931      0.852
        non-neoplastic         39         45      0.742      0.622      0.691      0.569
Speed: 4.7ms preprocess, 134.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val84
mAP@0.5: 0.8112
mAP@0.5:0.95: 0.7105
Precision: 0.8972
Recall: 0.9000
F1-score: 0.8986


In [10]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_s_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1353.1260.7 MB/s, size: 290.8 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:12<00:00,  1.73s/it]


                   all        100        115      0.801      0.813      0.843      0.728
            neoplastic         68         70      0.912      0.893      0.947       0.85
        non-neoplastic         39         45       0.69      0.733      0.739      0.606
Speed: 4.9ms preprocess, 61.9ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val85
mAP@0.5: 0.8431
mAP@0.5:0.95: 0.7279
Precision: 0.9124
Recall: 0.8929
F1-score: 0.9025


In [11]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_n_baseline_model_CIoU\yolov8_n_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1388.3124.7 MB/s, size: 247.3 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:07<00:00,  1.11s/it]


                   all        100        115      0.766      0.803      0.823      0.696
            neoplastic         68         70      0.789      0.963      0.924      0.825
        non-neoplastic         39         45      0.743      0.642      0.722      0.567
Speed: 2.7ms preprocess, 19.3ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs\detect\val86
mAP@0.5: 0.8231
mAP@0.5:0.95: 0.6961
Precision: 0.7893
Recall: 0.9634
F1-score: 0.8677


## YOLOv12 (CIoU + No Augmentations + No Negatives) for 100 test images ................(BKAI-IGH NeoPolyp Dataset)...................

In [12]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_l_baseline_model_CIoU\yolov12_l_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,340,614 parameters, 0 gradients, 88.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 784.3193.7 MB/s, size: 260.3 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:47<00:00,  6.82s/it]


                   all        100        115      0.715      0.824      0.786       0.68
            neoplastic         68         70      0.737      0.943      0.906      0.833
        non-neoplastic         39         45      0.694      0.705      0.667      0.527
Speed: 4.3ms preprocess, 416.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val87
mAP@0.5: 0.7863
mAP@0.5:0.95: 0.6796
Precision: 0.7369
Recall: 0.9429
F1-score: 0.8273


In [1]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_m_baseline_model_CIoU\yolov12_m_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,106,454 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1414.6261.8 MB/s, size: 273.0 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:01<00:00,  4.46it/s]


                   all        100        115      0.748      0.859       0.83      0.687
            neoplastic         68         70      0.816      0.914      0.926      0.807
        non-neoplastic         39         45       0.68      0.804      0.735      0.567
Speed: 0.5ms preprocess, 8.3ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs\detect\val91
mAP@0.5: 0.8302
mAP@0.5:0.95: 0.6870
Precision: 0.8164
Recall: 0.9143
F1-score: 0.8626


In [2]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_s_baseline_model_CIoU\yolov12_s_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,654 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1199.0375.4 MB/s, size: 310.8 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.13it/s]


                   all        100        115      0.761      0.803      0.839      0.738
            neoplastic         68         70      0.836      0.929      0.937      0.858
        non-neoplastic         39         45      0.685      0.678       0.74      0.619
Speed: 0.8ms preprocess, 6.2ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs\detect\val92
mAP@0.5: 0.8385
mAP@0.5:0.95: 0.7384
Precision: 0.8362
Recall: 0.9286
F1-score: 0.8799


In [3]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_n_baseline_model_CIoU\yolov12_n_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,557,118 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1392.5164.8 MB/s, size: 297.3 KB)


val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:06<00:00,  1.13it/s]


                   all        100        115      0.808      0.752      0.799      0.677
            neoplastic         68         70      0.903      0.843      0.922      0.834
        non-neoplastic         39         45      0.713      0.662      0.675       0.52
Speed: 0.6ms preprocess, 6.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to runs\detect\val93
mAP@0.5: 0.7986
mAP@0.5:0.95: 0.6771
Precision: 0.9027
Recall: 0.8429
F1-score: 0.8717


In [13]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class PolypDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_filenames = sorted(os.listdir(images_dir))
        self.mask_filenames = sorted(os.listdir(masks_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.masks_dir, self.mask_filenames[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Assuming masks are grayscale

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask

# Example transforms - resize to 512x512, convert to tensor, normalize images
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # Using ImageNet mean/std for RGB normalization
                         std=[0.229, 0.224, 0.225]),
])

# Usage example for Kvasir-Seg dataset
kvasir_images_dir = r"C:\Medical_image_analysis\Polyp_domain_adaptation\kvasir-seg\images"
kvasir_masks_dir = r"C:\Medical_image_analysis\Polyp_domain_adaptation\kvasir-seg\images"

kvasir_dataset = PolypDataset(kvasir_images_dir, kvasir_masks_dir, transform=transform)


In [14]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F

def visualize_samples(dataset, num_samples=5):
    for i in range(num_samples):
        image, mask = dataset[i]
        # Undo normalization for visualization
        image = image.clone()
        image = image * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        image = image + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        image = image.permute(1, 2, 0).numpy()
        image = (image * 255).astype('uint8')

        mask = mask.squeeze(0).numpy()  # Assuming mask is 1 channel grayscale

        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1)
        plt.title("Image")
        plt.imshow(image)
        plt.axis('off')

        plt.subplot(1,2,2)
        plt.title("Mask")
        plt.imshow(mask, cmap='gray')
        plt.axis('off')

        plt.show()

# Example usage after dataset is loaded
visualize_samples(kvasir_dataset, num_samples=5)


RuntimeError: output with shape [1, 512, 512] doesn't match the broadcast shape [3, 512, 512]

In [15]:
import os
import json
import numpy as np

# Paths
kvasir_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\raw\kvasir_on_combined"
bkai_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\raw\bkai_on_combined"
output_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused"

os.makedirs(output_path, exist_ok=True)

# Set weights for weighted average
weight_kvasir = 0.6   # change as per your choice
weight_bkai = 0.4     # sum of weights can be 1 or any numbers

# Get all JSON files (assuming same filenames in both folders)
json_files = sorted(os.listdir(kvasir_path))

for fname in json_files:
    kvasir_file = os.path.join(kvasir_path, fname)
    bkai_file = os.path.join(bkai_path, fname)

    # Load JSON predictions
    with open(kvasir_file, 'r') as f:
        kvasir_pred = json.load(f)

    with open(bkai_file, 'r') as f:
        bkai_pred = json.load(f)

    fused_pred = kvasir_pred.copy()
    fused_pred['preds'] = []

    # Assuming number of predictions (boxes) are the same and in same order
    for kv_box, bk_box in zip(kvasir_pred['preds'], bkai_pred['preds']):
        probs_kv = np.array(kv_box['probs'])
        probs_bk = np.array(bk_box['probs'])

        # Weighted average
        fused_probs = (weight_kvasir * probs_kv + weight_bkai * probs_bk) / (weight_kvasir + weight_bkai)

        # Copy bbox and class from one model (you can also implement bbox fusion later)
        fused_box = {
            'xyxy': kv_box['xyxy'],
            'conf': float(np.max(fused_probs)),  # new confidence = max prob
            'cls': int(np.argmax(fused_probs)),  # predicted class
            'probs': fused_probs.tolist()
        }

        fused_pred['preds'].append(fused_box)

    # Save fused JSON
    out_file = os.path.join(output_path, fname)
    with open(out_file, 'w') as f:
        json.dump(fused_pred, f, indent=2)

print(f"Fused predictions saved in {output_path}")


Fused predictions saved in C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused


In [21]:
import os
import json
import cv2

# Paths
fused_pred_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused"
image_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\combined_test\images"
output_vis_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\visualizations"

os.makedirs(output_vis_path, exist_ok=True)

# Colors for classes (example for 5 classes)
colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), (255,0,255)]

# List all images in the folder
all_images = {os.path.splitext(f)[0]: f for f in os.listdir(image_path)}

# List all fused JSONs
json_files = sorted(os.listdir(fused_pred_path))

for fname in json_files:
    # Load fused predictions
    with open(os.path.join(fused_pred_path, fname), 'r') as f:
        data = json.load(f)

    # Remove .json extension
    base_name = os.path.splitext(fname)[0]

    # Further remove .jpeg/.jpg/.png if present in JSON name
    for ext in ['.jpeg', '.jpg', '.png']:
        if base_name.lower().endswith(ext):
            base_name = base_name[:-len(ext)]

    # Find corresponding image
    img_file_name = all_images.get(base_name)
    if img_file_name is None:
        print(f"Image not found for {fname}, skipping.")
        continue

    img_file = os.path.join(image_path, img_file_name)
    img = cv2.imread(img_file)

    # Draw boxes
    for pred in data['preds']:
        xyxy = pred['xyxy']  # [x1, y1, x2, y2]
        cls = pred['cls']
        conf = pred['conf']
        color = colors[cls % len(colors)]
        cv2.rectangle(img, (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3])), color, 2)
        text = f"Class:{cls} Conf:{conf:.2f}"
        cv2.putText(img, text, (int(xyxy[0]), int(xyxy[1])-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Save visualization
    out_file = os.path.join(output_vis_path, os.path.splitext(fname)[0] + '.jpg')
    cv2.imwrite(out_file, img)

print(f"Visualization images saved in {output_vis_path}")


Visualization images saved in C:\Medical_image_analysis\Polyp_domain_adaptation\preds\visualizations
